<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_1_Foundations_Python_and_Math/Month_02_Data_Analysis_with_Pandas_and_NumPy/Week_2_Advanced_NumPy/Day_14_Week_2_Review_and_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/user-attachments/assets/2a4adf95-3d28-41e2-90d0-d06b2e9c8fa3" alt="Learn Data Science with Emmanuel Odenyire" width="180"/>

# Day 14: Week 2 Review and Advanced NumPy Project

## Phase 1 | Month 2 | Week 2 | Day 14

**Goal:** Consolidate all Week 2 knowledge and build a complete data science project using only NumPy.

### Week 2 Recap
| Day | Topic | Key Concept |
|-----|-------|-------------|
| 8 | Linear Algebra | Matrix multiplication, solve Ax=b |
| 9 | Vectorisation | np.where, speedup |
| 10 | Advanced Indexing | Fancy indexing, embeddings |
| 11 | Applications | Image processing, time series, pipeline |
| 12 | File I/O | .npy, .npz, genfromtxt |
| 13 | Statistics | PCA, permutation tests, correlation |

### Resources
- 🔢 [NumPy 100 Exercises (do 40-70)](https://github.com/rougier/numpy-100)
- 📖 [Python Data Science Handbook — Free Online](https://jakevdp.github.io/PythonDataScienceHandbook/)
- 🎥 [SciPy 2019 NumPy Tutorial](https://www.youtube.com/watch?v=ZB7BZMhfPgk)

In [ ]:
import numpy as np
np.random.seed(42)
print('=== Week 2 Quick Reference ===')

# Linear algebra
A = np.random.randn(3, 3)
print('det:', round(np.linalg.det(A), 3))
print('inv check:', np.allclose(A @ np.linalg.inv(A), np.eye(3)))

# Vectorised conditional
x = np.array([-3, -1, 0, 2, 5])
print('ReLU:', np.where(x > 0, x, 0))

# Fancy indexing
m = np.arange(25).reshape(5, 5)
print('Fancy rows [0,2,4]:', m[[0,2,4]].shape)

# Stats
data = np.random.randn(100)
print(f'Stats: mean={data.mean():.3f} p25={np.percentile(data,25):.3f}')
print('Week 2 ready!')


## Major Project: Nairobi Real Estate Price Predictor

**Scenario:** You are a data scientist at a Nairobi property portal. Build a complete regression pipeline using only NumPy.

**Steps:**
1. Generate realistic housing data
2. Train/val/test split
3. Preprocessing pipeline
4. Linear regression (normal equations)
5. Evaluation with RMSE and R²
6. Save model to disk

In [ ]:
import numpy as np
np.random.seed(2024)

n = 500
bedrooms    = np.random.randint(1, 6, n).astype(float)
bathrooms   = np.clip(bedrooms*0.7 + np.random.randn(n)*0.5, 1, 5).round()
size_sqft   = bedrooms * 400 + np.random.normal(0, 200, n)
age_years   = np.random.randint(0, 40, n).astype(float)
dist_cbd    = np.random.exponential(10, n).clip(0.5, 50)
neighb      = np.random.choice([1,2,3,4,5], n).astype(float)
noise       = np.random.normal(0, 0.5, n)
price       = (bedrooms*2.5 + bathrooms*1.5 + size_sqft*0.005
               - age_years*0.05 - dist_cbd*0.3 + neighb*0.8 + noise)
price       = np.clip(price, 2.0, 60.0)

X_raw  = np.column_stack([bedrooms, bathrooms, size_sqft, age_years, dist_cbd, neighb])
y      = price
fnames = ['Bedrooms','Bathrooms','Size(sqft)','Age(yrs)','CBD Dist','Neighbourhood']

print('=== Nairobi Housing Dataset ===')
print(f'Samples: {n}  Features: {X_raw.shape[1]}')
for name, col in zip(fnames, X_raw.T):
    print(f'  {name:15}: mean={col.mean():6.1f}  std={col.std():5.1f}')
print(f'\nPrice (MKES): mean={y.mean():.2f}  min={y.min():.1f}  max={y.max():.1f}')

In [ ]:
import numpy as np
np.random.seed(2024)

# Recreate data
n = 500
bedrooms = np.random.randint(1,6,n).astype(float)
bathrooms = np.clip(bedrooms*0.7+np.random.randn(n)*0.5,1,5).round()
size_sqft = bedrooms*400+np.random.normal(0,200,n)
age_years = np.random.randint(0,40,n).astype(float)
dist_cbd  = np.random.exponential(10,n).clip(0.5,50)
neigjb    = np.random.choice([1,2,3,4,5],n).astype(float)
noise     = np.random.normal(0,0.5,n)
price     = (bedrooms*2.5+bathrooms*1.5+size_sqft*0.005
             -age_years*0.05-dist_cbd*0.3+neigjb*0.8+noise)
price     = np.clip(price, 2.0, 60.0)
X_raw     = np.column_stack([bedrooms,bathrooms,size_sqft,age_years,dist_cbd,neigjb])
y         = price

# Train/val/test split
idx   = np.random.permutation(n)
X_tr, y_tr = X_raw[idx[:350]], y[idx[:350]]
X_v,  y_v  = X_raw[idx[350:425]], y[idx[350:425]]
X_te, y_te = X_raw[idx[425:]], y[idx[425:]]

# Preprocessing
mean = X_tr.mean(0); std = X_tr.std(0)
X_tr_s = (X_tr - mean)/std
X_v_s  = (X_v  - mean)/std
X_te_s = (X_te - mean)/std

# Normal equation
X_aug = np.hstack([np.ones((X_tr_s.shape[0],1)), X_tr_s])
theta, _, _, _ = np.linalg.lstsq(X_aug, y_tr, rcond=None)

def predict(Xs):
    return np.hstack([np.ones((Xs.shape[0],1)), Xs]) @ theta

def rmse(yt, yp): return np.sqrt(np.mean((yt-yp)**2))
def r2(yt, yp):
    return 1 - np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2)

print('=== Model Results ===')
for name, Xs, yt in [('Train', X_tr_s, y_tr), ('Val', X_v_s, y_v), ('Test', X_te_s, y_te)]:
    yp = predict(Xs)
    print(f'{name:5}: RMSE={rmse(yt,yp):.3f} MKES  R2={r2(yt,yp):.4f}')

fnames = ['Intercept','Bedrooms','Bathrooms','Size','Age','CBD Dist','Neighbourhood']
print('\nCoefficients:')
for n_, c in zip(fnames, theta):
    print(f'  {n_:15}: {c:+.4f}')

## Week 2 Challenges

1. **Polynomial Features:** Add size², bedrooms², and interaction terms. Does R² improve?
2. **Ridge Regression:** Add L2 regularisation `lambda * ||theta||²` to the normal equations
3. **Bootstrap CI:** Use 1000 bootstrap samples to compute 95% CI for RMSE on the test set
4. **NumPy 100:** Complete exercises 40–70 from https://github.com/rougier/numpy-100

In [ ]:
import numpy as np
# Extend the model with polynomial features and regularisation
# YOUR CODE HERE

## Week 2 Complete!

### Self-Assessment Checklist
- [ ] I can solve Ax=b using np.linalg.solve
- [ ] I replace loops with vectorised operations naturally
- [ ] I use embedding lookup for NLP/RecSys problems
- [ ] I save/load datasets in .npz format
- [ ] I can implement PCA from scratch
- [ ] I completed the real estate project (R² > 0.85)

### Week 3 Preview: Pandas Basics
Now that you master NumPy (the engine), it's time for Pandas (the car). Pandas wraps NumPy with labelled columns, flexible indexing, and powerful data manipulation. Every company uses Pandas for data analysis.